<a href="https://colab.research.google.com/github/adityamishara051-lab/AdityaKumar-2026B03110282/blob/main/2026B03110282_Aditya_Kumar_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 7: Building Energy Consumption Analysis
**Industry:** Energy / Electrical Engineering  
**Student Name:** Aditya Kumar  
**Roll Number:** 2026B03110282  
**Program:** Five-Day Engineering Student Training | First Year – First Semester  
**Dataset:** `07_energy_consumption.csv`  

---

### Business Scenario & Analytical Approach
Analyze building electricity consumption records to identify high-usage buildings, temporal and seasonal consumption patterns, peak vs. off-peak load behavior, per-occupant efficiency, and cost-saving opportunities. All metrics are computed dynamically from the dataset without hard-coding, and findings are interpreted within the limitations of a synthetic teaching dataset (correlation does not imply causation).

## Task 1: Load and Inspect Energy Records
Load `07_energy_consumption.csv` using a Google Colab–compatible loader and inspect dimensions, data types, duplicate counts, and summary statistics.

In [1]:
import os
import numpy as np
import pandas as pd

# Configure clean tabular display in Google Colab
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

def load_dataset(filepath):
    """Loads the CSV dataset with fallback paths for Google Colab or GitHub repo folders."""
    possible_paths = [
        filepath,
        os.path.join('data', os.path.basename(filepath)),
        os.path.join('..', 'data', os.path.basename(filepath))
    ]
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Dataset loaded successfully from: {path}")
            return pd.read_csv(path)
    raise FileNotFoundError(f"Could not find {filepath}. Please upload '07_energy_consumption.csv' to Colab.")

df_raw = load_dataset('07_energy_consumption.csv')

print(f"\nDataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print("\nColumn Data Types:")
print(df_raw.dtypes)
print("\nUnique Buildings:", df_raw['Building'].unique().tolist())
print("Duplicate Rows:", df_raw.duplicated().sum())

print("\nFirst 5 Records:")
print(df_raw.head().to_string())

print("\nDescriptive Statistics:")
print(df_raw.describe(include='all').to_string())

Dataset loaded successfully from: 07_energy_consumption.csv

Dataset Shape: 600 rows x 7 columns

Column Data Types:
Date                object
Building            object
Units_kWh          float64
Peak_Hour_Units    float64
Off_Peak_Units     float64
Temperature_C      float64
Occupancy            int64
dtype: object

Unique Buildings: ['Block C', 'Block A', 'Block B', 'Lab']
Duplicate Rows: 0

First 5 Records:
         Date Building  Units_kWh  Peak_Hour_Units  Off_Peak_Units  Temperature_C  Occupancy
0  2025-01-01  Block C     425.80           238.30          276.10          37.30        217
1  2025-01-02  Block A     332.80           167.40          249.50          18.10         41
2  2025-01-03  Block B     307.50           136.90          361.20          38.80         61
3  2025-01-04      Lab     398.70           104.70          417.00          36.00        280
4  2025-01-05  Block A     531.30           197.20          186.40          30.20         64

Descriptive Statistics:
 

## Task 2: Identify and Handle Missing `Units_kWh` Values
Inspect missing values across all columns and impute missing `Units_kWh` values using each building's **median** consumption (`groupby('Building')['Units_kWh'].transform('median')`) so building-specific load distributions are preserved without distortion from extreme values.

In [ ]:
# Check missing values across all columns
missing_summary = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
})
print("Missing Values Summary Before Cleaning:")
print(missing_summary.to_string())

print("\nRecords with Missing Units_kWh:")
print(df_raw[df_raw['Units_kWh'].isnull()].to_string())

# Impute missing Units_kWh using Building-wise median
df = df_raw.copy()
building_medians = df.groupby('Building')['Units_kWh'].median().to_frame(name='Median_Units_kWh')
print("\nBuilding-wise Median Units_kWh Used for Imputation:")
print(building_medians.to_string())

df['Units_kWh'] = df['Units_kWh'].fillna(df.groupby('Building')['Units_kWh'].transform('median'))

print("\nMissing Values Count After Cleaning:")
print(df.isnull().sum().to_string())

Missing Values Summary Before Cleaning:
                 Missing_Count  Missing_Percentage (%)
Date                         0                    0.00
Building                     0                    0.00
Units_kWh                   10                    1.67
Peak_Hour_Units              0                    0.00
Off_Peak_Units               0                    0.00
Temperature_C                0                    0.00
Occupancy                    0                    0.00

Records with Missing Units_kWh:
           Date Building  Units_kWh  Peak_Hour_Units  Off_Peak_Units  Temperature_C  Occupancy
80   2025-03-22  Block A        NaN           117.00          186.40          35.50        264
101  2025-04-12  Block C        NaN           181.50          366.50          33.30        255
239  2025-08-28  Block B        NaN           155.70          241.40          30.40        103
312  2025-11-09      Lab        NaN           171.50          261.80          28.20        140
358  2025-12

## Task 3: Convert `Date` to Datetime and Extract Month and Weekday
Parse the `Date` column into Pandas `datetime64` format and extract `Year`, `Month`, `Month_Name`, `Weekday`, and `Day_Type` (`Weekday` vs. `Weekend`).

In [ ]:
# Convert Date to datetime and extract temporal attributes
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Weekday'] = df['Date'].dt.day_name()
df['Day_Type'] = np.where(df['Date'].dt.dayofweek >= 5, 'Weekend', 'Weekday')

print(f"Date Range Covered: {df['Date'].min().date()} to {df['Date'].max().date()}")
print("\nSample Records with Extracted Date Features:")
print(df[['Date', 'Year', 'Month', 'Month_Name', 'Weekday', 'Day_Type', 'Building', 'Units_kWh']].head(10).to_string())

Date Range Covered: 2025-01-01 to 2026-08-23

Sample Records with Extracted Date Features:
        Date  Year  Month Month_Name    Weekday Day_Type Building  Units_kWh
0 2025-01-01  2025      1        Jan  Wednesday  Weekday  Block C     425.80
1 2025-01-02  2025      1        Jan   Thursday  Weekday  Block A     332.80
2 2025-01-03  2025      1        Jan     Friday  Weekday  Block B     307.50
3 2025-01-04  2025      1        Jan   Saturday  Weekend      Lab     398.70
4 2025-01-05  2025      1        Jan     Sunday  Weekend  Block A     531.30
5 2025-01-06  2025      1        Jan     Monday  Weekday  Block A     337.60
6 2025-01-07  2025      1        Jan    Tuesday  Weekday  Block B     479.60
7 2025-01-08  2025      1        Jan  Wednesday  Weekday  Block A     429.50
8 2025-01-09  2025      1        Jan   Thursday  Weekday  Block C     619.30
9 2025-01-10  2025      1        Jan     Friday  Weekday  Block A     459.00


## Task 4: Calculate Total and Average Energy Consumption
Compute overall total electricity consumption (`Units_kWh`), record-level mean consumption, minimum, and maximum values across the entire dataset.

In [ ]:
total_kwh = df['Units_kWh'].sum()
avg_kwh = df['Units_kWh'].mean()
min_kwh = df['Units_kWh'].min()
max_kwh = df['Units_kWh'].max()

overall_summary = pd.DataFrame({
    'Metric': [
        'Total Records Analyzed',
        'Total Energy Consumption (kWh)',
        'Average Consumption per Record (kWh)',
        'Minimum Single-Record Consumption (kWh)',
        'Maximum Single-Record Consumption (kWh)'
    ],
    'Value': [
        f"{len(df):,}",
        f"{total_kwh:,.2f}",
        f"{avg_kwh:,.2f}",
        f"{min_kwh:,.2f}",
        f"{max_kwh:,.2f}"
    ]
})
print("Overall Facility Energy Consumption Summary:")
print(overall_summary.to_string(index=False))

Overall Facility Energy Consumption Summary:
                                 Metric      Value
                 Total Records Analyzed        600
         Total Energy Consumption (kWh) 257,714.05
   Average Consumption per Record (kWh)     429.52
Minimum Single-Record Consumption (kWh)     109.70
Maximum Single-Record Consumption (kWh)     708.90


## Task 5: Rank Buildings by Total Energy Use
Group records by `Building` to compute total kWh, percentage share of total consumption, record count, mean, median, and peak record usage, ranked from highest to lowest.

In [ ]:
building_ranking = (
    df.groupby('Building', as_index=False)
      .agg(
          Record_Count=('Units_kWh', 'count'),
          Total_Units_kWh=('Units_kWh', 'sum'),
          Average_Units_kWh=('Units_kWh', 'mean'),
          Median_Units_kWh=('Units_kWh', 'median'),
          Max_Units_kWh=('Units_kWh', 'max')
      )
      .sort_values(by='Total_Units_kWh', ascending=False)
      .reset_index(drop=True)
)

building_ranking.insert(0, 'Rank', building_ranking.index + 1)
building_ranking['Share_of_Total (%)'] = (building_ranking['Total_Units_kWh'] / total_kwh) * 100

print("Building Ranking by Total Energy Use:")
print(building_ranking.to_string(index=False))

Building Ranking by Total Energy Use:
 Rank Building  Record_Count  Total_Units_kWh  Average_Units_kWh  Median_Units_kWh  Max_Units_kWh  Share_of_Total (%)
    1  Block A           157        68,330.10             435.22            429.90         666.20               26.51
    2      Lab           157        67,600.55             430.58            434.85         664.70               26.23
    3  Block B           146        61,158.80             418.90            423.00         708.90               23.73
    4  Block C           140        60,624.60             433.03            448.10         683.70               23.52


## Task 6: Find the Highest-Consumption Dates
Identify the top 10 highest-consumption dates along with building name, weekday, peak/off-peak split, ambient temperature, and occupancy.

In [ ]:
top_10_dates = (
    df[['Date', 'Weekday', 'Building', 'Units_kWh', 'Peak_Hour_Units', 'Off_Peak_Units', 'Temperature_C', 'Occupancy']]
    .sort_values(by='Units_kWh', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_10_dates.insert(0, 'Rank', top_10_dates.index + 1)

print("Top 10 Highest-Consumption Dates:")
print(top_10_dates.to_string(index=False))

Top 10 Highest-Consumption Dates:
 Rank       Date  Weekday Building  Units_kWh  Peak_Hour_Units  Off_Peak_Units  Temperature_C  Occupancy
    1 2025-06-05 Thursday  Block B     708.90           121.10          302.70          27.10         69
    2 2026-04-18 Saturday  Block C     683.70           173.90          262.50          23.70         86
    3 2026-05-05  Tuesday  Block A     666.20           113.00          412.40          20.10        258
    4 2025-07-15  Tuesday      Lab     664.70           207.30          247.10          18.10         45
    5 2025-06-02   Monday  Block C     662.40           203.90          319.80          23.30         56
    6 2025-07-27   Sunday  Block B     656.30           181.80          343.40          21.00         54
    7 2026-02-05 Thursday  Block C     654.30           136.00          159.20          29.10        269
    8 2025-03-11  Tuesday  Block B     646.40           148.20          140.90          25.00         48
    9 2026-07-27   Mo

## Task 7: Calculate Monthly Consumption by Building
Aggregate monthly electricity consumption by building using both long-form `groupby()` tables and a `Year_Month` × `Building` pivot matrix.

In [ ]:
# Grouped monthly consumption by building
monthly_building_long = (
    df.groupby(['Year', 'Month', 'Month_Name', 'Building'], as_index=False)['Units_kWh']
      .sum()
      .sort_values(by=['Year', 'Month', 'Building'])
)
print("Monthly Consumption by Building (First 16 Rows):")
print(monthly_building_long.head(16).to_string(index=False))

# Pivot matrix by Year-Month and Building
df['Year_Month'] = df['Date'].dt.to_period('M').astype(str)
monthly_pivot = df.pivot_table(
    index='Year_Month',
    columns='Building',
    values='Units_kWh',
    aggfunc='sum',
    fill_value=0
)
monthly_pivot['Monthly_Total_kWh'] = monthly_pivot.sum(axis=1)
print("\nMonthly Consumption Matrix (kWh) Across All Buildings:")
print(monthly_pivot.to_string())

Monthly Consumption by Building (First 16 Rows):
 Year  Month Month_Name Building  Units_kWh
 2025      1        Jan  Block A   5,416.10
 2025      1        Jan  Block B   1,096.60
 2025      1        Jan  Block C   2,511.70
 2025      1        Jan      Lab   3,903.10
 2025      2        Feb  Block A   1,728.10
 2025      2        Feb  Block B   3,257.90
 2025      2        Feb  Block C   2,278.50
 2025      2        Feb      Lab   4,475.30
 2025      3        Mar  Block A   3,436.70
 2025      3        Mar  Block B   5,038.90
 2025      3        Mar  Block C   3,628.20
 2025      3        Mar      Lab     910.00
 2025      4        Apr  Block A   3,411.80
 2025      4        Apr  Block B   2,118.80
 2025      4        Apr  Block C   3,207.00
 2025      4        Apr      Lab   4,567.30

Monthly Consumption Matrix (kWh) Across All Buildings:
Building    Block A  Block B  Block C      Lab  Monthly_Total_kWh
Year_Month                                                       
2025-01    5,41

## Task 8: Compare Peak-Hour and Off-Peak Usage
Compare total, average, and percentage shares of `Peak_Hour_Units` and `Off_Peak_Units` overall and across individual buildings.

In [ ]:
total_peak = df['Peak_Hour_Units'].sum()
total_off_peak = df['Off_Peak_Units'].sum()
combined_metered = total_peak + total_off_peak

peak_overall = pd.DataFrame({
    'Usage_Period': ['Peak-Hour Usage', 'Off-Peak Usage', 'Combined Metered (Peak + Off-Peak)'],
    'Total_Units_kWh': [total_peak, total_off_peak, combined_metered],
    'Average_Units_kWh': [
        df['Peak_Hour_Units'].mean(),
        df['Off_Peak_Units'].mean(),
        (df['Peak_Hour_Units'] + df['Off_Peak_Units']).mean()
    ],
    'Share (%)': [
        (total_peak / combined_metered) * 100,
        (total_off_peak / combined_metered) * 100,
        100.0
    ]
})
print("Overall Peak-Hour vs. Off-Peak Usage Comparison:")
print(peak_overall.to_string(index=False))

peak_by_building = (
    df.groupby('Building', as_index=False)
      .agg(
          Total_Peak_kWh=('Peak_Hour_Units', 'sum'),
          Total_Off_Peak_kWh=('Off_Peak_Units', 'sum'),
          Avg_Peak_kWh=('Peak_Hour_Units', 'mean'),
          Avg_Off_Peak_kWh=('Off_Peak_Units', 'mean')
      )
)
peak_by_building['Peak_Share (%)'] = (
    peak_by_building['Total_Peak_kWh'] /
    (peak_by_building['Total_Peak_kWh'] + peak_by_building['Total_Off_Peak_kWh'])
) * 100
peak_by_building['Off_Peak_Share (%)'] = 100.0 - peak_by_building['Peak_Share (%)']

print("\nBuilding-wise Peak vs. Off-Peak Usage Breakdown:")
print(peak_by_building.to_string(index=False))

Overall Peak-Hour vs. Off-Peak Usage Comparison:
                      Usage_Period  Total_Units_kWh  Average_Units_kWh  Share (%)
                   Peak-Hour Usage        92,269.50             153.78      36.46
                    Off-Peak Usage       160,788.30             267.98      63.54
Combined Metered (Peak + Off-Peak)       253,057.80             421.76     100.00

Building-wise Peak vs. Off-Peak Usage Breakdown:
Building  Total_Peak_kWh  Total_Off_Peak_kWh  Avg_Peak_kWh  Avg_Off_Peak_kWh  Peak_Share (%)  Off_Peak_Share (%)
 Block A       23,942.40           41,813.40        152.50            266.33           36.41               63.59
 Block B       21,956.10           38,547.10        150.38            264.02           36.29               63.71
 Block C       21,795.10           38,220.60        155.68            273.00           36.32               63.68
     Lab       24,575.90           42,207.20        156.53            268.84           36.80               63.20


## Task 9: Calculate Average Consumption per Occupant
Compute energy intensity per occupant (`kWh_per_Occupant = Units_kWh / Occupancy`) at both record-level and occupancy-weighted building levels.

In [ ]:
df['kWh_per_Occupant'] = np.where(df['Occupancy'] > 0, df['Units_kWh'] / df['Occupancy'], np.nan)

mean_rec_per_occ = df['kWh_per_Occupant'].mean()
weighted_per_occ = df['Units_kWh'].sum() / df['Occupancy'].sum()

print(f"Mean Record-Level Consumption per Occupant: {mean_rec_per_occ:.2f} kWh/occupant")
print(f"Occupancy-Weighted Facility Intensity (Total kWh / Total Occupancy): {weighted_per_occ:.2f} kWh/occupant")

occupant_efficiency = (
    df.groupby('Building', as_index=False)
      .agg(
          Total_Units_kWh=('Units_kWh', 'sum'),
          Total_Occupancy=('Occupancy', 'sum'),
          Avg_Occupancy=('Occupancy', 'mean'),
          Avg_kWh_per_Occupant=('kWh_per_Occupant', 'mean')
      )
)
occupant_efficiency['Weighted_kWh_per_Occupant'] = (
    occupant_efficiency['Total_Units_kWh'] / occupant_efficiency['Total_Occupancy']
)
occupant_efficiency = occupant_efficiency.sort_values(by='Weighted_kWh_per_Occupant', ascending=False).reset_index(drop=True)

print("\nBuilding-wise Energy Consumption per Occupant:")
print(occupant_efficiency.to_string(index=False))

Mean Record-Level Consumption per Occupant: 4.39 kWh/occupant
Occupancy-Weighted Facility Intensity (Total kWh / Total Occupancy): 2.75 kWh/occupant

Building-wise Energy Consumption per Occupant:
Building  Total_Units_kWh  Total_Occupancy  Avg_Occupancy  Avg_kWh_per_Occupant  Weighted_kWh_per_Occupant
 Block A        68,330.10            22825         145.38                  4.73                       2.99
 Block C        60,624.60            22061         157.58                  4.52                       2.75
 Block B        61,158.80            22287         152.65                  4.52                       2.74
     Lab        67,600.55            26639         169.68                  3.80                       2.54


## Task 10: Use NumPy to Calculate Median, Variance, and Standard Deviation
Convert numerical columns into native NumPy arrays (`np.ndarray`) to compute mean, median, sample/population variance, standard deviation, and quartiles.

In [ ]:
units_arr = df['Units_kWh'].to_numpy()
peak_arr = df['Peak_Hour_Units'].to_numpy()
off_peak_arr = df['Off_Peak_Units'].to_numpy()
temp_arr = df['Temperature_C'].to_numpy()
occ_arr = df['Occupancy'].to_numpy()

metrics_dict = {
    'Units_kWh': units_arr,
    'Peak_Hour_Units': peak_arr,
    'Off_Peak_Units': off_peak_arr,
    'Temperature_C': temp_arr,
    'Occupancy': occ_arr
}

numpy_stats_rows = []
for col_name, arr in metrics_dict.items():
    numpy_stats_rows.append({
        'Variable': col_name,
        'Mean': np.mean(arr),
        'Median': np.median(arr),
        'Sample_Variance (ddof=1)': np.var(arr, ddof=1),
        'Sample_Std_Dev (ddof=1)': np.std(arr, ddof=1),
        'Pop_Variance (ddof=0)': np.var(arr, ddof=0),
        'Pop_Std_Dev (ddof=0)': np.std(arr, ddof=0),
        'Q1_25th': np.percentile(arr, 25),
        'Q3_75th': np.percentile(arr, 75)
    })

numpy_stats_df = pd.DataFrame(numpy_stats_rows)
print("NumPy Statistical Summary Table:")
print(numpy_stats_df.to_string(index=False))

NumPy Statistical Summary Table:
       Variable   Mean  Median  Sample_Variance (ddof=1)  Sample_Std_Dev (ddof=1)  Pop_Variance (ddof=0)  Pop_Std_Dev (ddof=0)  Q1_25th  Q3_75th
      Units_kWh 429.52  430.00                 10,533.31                   102.63              10,515.76                102.55   356.25   502.10
Peak_Hour_Units 153.78  152.20                  1,735.62                    41.66               1,732.72                 41.63   124.80   179.72
 Off_Peak_Units 267.98  266.85                  4,568.23                    67.59               4,560.61                 67.53   223.62   315.30
  Temperature_C  28.97   29.00                     38.60                     6.21                  38.53                  6.21    24.70    33.20
      Occupancy 156.35  154.50                  6,932.98                    83.26               6,921.43                 83.20    82.75   232.00


## Task 11: Identify High-Consumption Outliers Using IQR
Calculate $Q1$, $Q3$, $IQR = Q3 - Q1$, and upper/lower outlier fences ($Q3 + 1.5 \times IQR$) using `np.percentile()` and inspect extreme consumption records.

In [ ]:
q1 = np.percentile(units_arr, 25)
q3 = np.percentile(units_arr, 75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

iqr_summary = pd.DataFrame({
    'IQR_Metric': ['Q1 (25th Percentile)', 'Q3 (75th Percentile)', 'IQR (Q3 - Q1)', 'Lower Fence (Q1 - 1.5*IQR)', 'Upper Fence (Q3 + 1.5*IQR)'],
    'Value_kWh': [q1, q3, iqr, lower_fence, upper_fence]
})
print("IQR Outlier Detection Thresholds for Units_kWh:")
print(iqr_summary.to_string(index=False))

high_outlier_mask = units_arr > upper_fence
low_outlier_mask = units_arr < lower_fence

high_outliers_df = df[high_outlier_mask].sort_values(by='Units_kWh', ascending=False)
print(f"\nHigh-Consumption Outliers Count (Units_kWh > {upper_fence:.2f}): {len(high_outliers_df)}")
print(f"Low-Consumption Outliers Count (Units_kWh < {lower_fence:.2f}): {low_outlier_mask.sum()}")

if len(high_outliers_df) > 0:
    print(high_outliers_df[['Date', 'Building', 'Units_kWh', 'Temperature_C', 'Occupancy']].to_string(index=False))
else:
    print("\nNo records exceed the strict upper IQR fence (Q3 + 1.5*IQR). Top 5 extreme records closest to the upper fence:")
    print(df.sort_values(by='Units_kWh', ascending=False)[['Date', 'Building', 'Units_kWh', 'Temperature_C', 'Occupancy']].head(5).to_string(index=False))

IQR Outlier Detection Thresholds for Units_kWh:
                IQR_Metric  Value_kWh
      Q1 (25th Percentile)     356.25
      Q3 (75th Percentile)     502.10
             IQR (Q3 - Q1)     145.85
Lower Fence (Q1 - 1.5*IQR)     137.47
Upper Fence (Q3 + 1.5*IQR)     720.88

High-Consumption Outliers Count (Units_kWh > 720.88): 0
Low-Consumption Outliers Count (Units_kWh < 137.47): 1

No records exceed the strict upper IQR fence (Q3 + 1.5*IQR). Top 5 extreme records closest to the upper fence:
      Date Building  Units_kWh  Temperature_C  Occupancy
2025-06-05  Block B     708.90          27.10         69
2026-04-18  Block C     683.70          23.70         86
2026-05-05  Block A     666.20          20.10        258
2025-07-15      Lab     664.70          18.10         45
2025-06-02  Block C     662.40          23.30         56


## Task 12: Compare Consumption with Temperature Using Correlation
Evaluate Pearson correlation between `Units_kWh` and `Temperature_C` overall and by building using both Pandas `.corr()` and NumPy `np.corrcoef()`. *(Note: Correlation measures linear association and does not prove causation.)*

In [ ]:
pearson_pandas = df['Units_kWh'].corr(df['Temperature_C'])
pearson_numpy = np.corrcoef(units_arr, temp_arr)[0, 1]

print(f"Overall Pearson Correlation (Units_kWh vs Temperature_C) [Pandas]: {pearson_pandas:.4f}")
print(f"Overall Pearson Correlation (Units_kWh vs Temperature_C) [NumPy]:  {pearson_numpy:.4f}")

building_corrs = []
for bldg, group in df.groupby('Building'):
    building_corrs.append({
        'Building': bldg,
        'Temp_vs_Units_Corr': group['Units_kWh'].corr(group['Temperature_C']),
        'Mean_Temp_C': group['Temperature_C'].mean(),
        'Mean_Units_kWh': group['Units_kWh'].mean()
    })

print("\nBuilding-wise Correlation (Units_kWh vs Temperature_C):")
print(pd.DataFrame(building_corrs).to_string(index=False))

num_cols = ['Units_kWh', 'Peak_Hour_Units', 'Off_Peak_Units', 'Temperature_C', 'Occupancy']
print("\nFull Numerical Correlation Matrix:")
print(df[num_cols].corr().to_string())

Overall Pearson Correlation (Units_kWh vs Temperature_C) [Pandas]: -0.0697
Overall Pearson Correlation (Units_kWh vs Temperature_C) [NumPy]:  -0.0697

Building-wise Correlation (Units_kWh vs Temperature_C):
Building  Temp_vs_Units_Corr  Mean_Temp_C  Mean_Units_kWh
 Block A               -0.16        28.36          435.22
 Block B               -0.07        29.18          418.90
 Block C                0.08        29.31          433.03
     Lab               -0.11        29.09          430.58

Full Numerical Correlation Matrix:
                 Units_kWh  Peak_Hour_Units  Off_Peak_Units  Temperature_C  Occupancy
Units_kWh             1.00             0.04            0.06          -0.07      -0.04
Peak_Hour_Units       0.04             1.00           -0.02          -0.03      -0.05
Off_Peak_Units        0.06            -0.02            1.00           0.06       0.08
Temperature_C        -0.07            -0.03            0.06           1.00      -0.02
Occupancy            -0.04           

## Task 13: Create High/Normal Consumption Label Using a Documented Threshold
**Documented Threshold Rule:** Records with `Units_kWh > 500.0 kWh` (~75th percentile) are labeled **`High`**, while records with `Units_kWh <= 500.0 kWh` are labeled **`Normal`**.

In [ ]:
CONSUMPTION_THRESHOLD_KWH = 500.0  # Documented threshold (> 500 kWh per record)

df['Consumption_Category'] = np.where(df['Units_kWh'] > CONSUMPTION_THRESHOLD_KWH, 'High', 'Normal')

category_summary = (
    df.groupby('Consumption_Category', as_index=False)
      .agg(
          Record_Count=('Units_kWh', 'count'),
          Total_Units_kWh=('Units_kWh', 'sum'),
          Average_Units_kWh=('Units_kWh', 'mean'),
          Avg_Temperature_C=('Temperature_C', 'mean'),
          Avg_Occupancy=('Occupancy', 'mean')
      )
)
category_summary['Record_Share (%)'] = (category_summary['Record_Count'] / len(df)) * 100
category_summary['Energy_Share (%)'] = (category_summary['Total_Units_kWh'] / total_kwh) * 100

print(f"Consumption Classification Summary (Threshold > {CONSUMPTION_THRESHOLD_KWH} kWh):")
print(category_summary.to_string(index=False))

bldg_cat_crosstab = pd.crosstab(df['Building'], df['Consumption_Category'], margins=True, margins_name='Total')
print("\nBuilding-wise High vs. Normal Consumption Record Counts:")
print(bldg_cat_crosstab.to_string())

Consumption Classification Summary (Threshold > 500.0 kWh):
Consumption_Category  Record_Count  Total_Units_kWh  Average_Units_kWh  Avg_Temperature_C  Avg_Occupancy  Record_Share (%)  Energy_Share (%)
                High           154        85,814.30             557.24              27.97         147.72             25.67             33.30
              Normal           446       171,899.75             385.43              29.32         159.33             74.33             66.70

Building-wise High vs. Normal Consumption Record Counts:
Consumption_Category  High  Normal  Total
Building                                 
Block A                 44     113    157
Block B                 30     116    146
Block C                 32     108    140
Lab                     48     109    157
Total                  154     446    600


## Task 14: Calculate Estimated Cost Using an Assumed Tariff per kWh
**Assumed Tariff Disclosure:** Since utility billing rates are not provided in the raw dataset, a flat commercial electricity tariff of **INR 8.50 per kWh** (`₹8.50/kWh`) is explicitly assumed to estimate facility electricity expenditure.

In [ ]:
ASSUMED_TARIFF_INR_PER_KWH = 8.50  # Explicitly assumed flat commercial tariff (INR 8.50 per kWh)

df['Estimated_Cost_INR'] = df['Units_kWh'] * ASSUMED_TARIFF_INR_PER_KWH

total_estimated_cost = df['Estimated_Cost_INR'].sum()
avg_daily_cost = df['Estimated_Cost_INR'].mean()

print(f"Assumed Commercial Tariff: INR {ASSUMED_TARIFF_INR_PER_KWH:.2f} per kWh")
print(f"Total Estimated Facility Electricity Cost: INR {total_estimated_cost:,.2f}")
print(f"Average Estimated Cost per Record: INR {avg_daily_cost:,.2f}")

cost_by_building = (
    df.groupby('Building', as_index=False)
      .agg(
          Total_Units_kWh=('Units_kWh', 'sum'),
          Estimated_Total_Cost_INR=('Estimated_Cost_INR', 'sum'),
          Average_Record_Cost_INR=('Estimated_Cost_INR', 'mean')
      )
      .sort_values(by='Estimated_Total_Cost_INR', ascending=False)
      .reset_index(drop=True)
)
cost_by_building['Cost_Share (%)'] = (cost_by_building['Estimated_Total_Cost_INR'] / total_estimated_cost) * 100

print("\nEstimated Cost Breakdown by Building (Assumed Tariff = INR 8.50/kWh):")
print(cost_by_building.to_string(index=False))

Assumed Commercial Tariff: INR 8.50 per kWh
Total Estimated Facility Electricity Cost: INR 2,190,569.42
Average Estimated Cost per Record: INR 3,650.95

Estimated Cost Breakdown by Building (Assumed Tariff = INR 8.50/kWh):
Building  Total_Units_kWh  Estimated_Total_Cost_INR  Average_Record_Cost_INR  Cost_Share (%)
 Block A        68,330.10                580,805.85                 3,699.40           26.51
     Lab        67,600.55                574,604.68                 3,659.90           26.23
 Block B        61,158.80                519,849.80                 3,560.62           23.73
 Block C        60,624.60                515,309.10                 3,680.78           23.52


## Task 15: Export Cleaned Dataset & Five Energy-Saving Recommendations
Export the cleaned, feature-enriched dataset (`cleaned_07_energy_consumption.csv`) and summarize five data-supported energy-saving recommendations.

In [ ]:
cleaned_filename = 'cleaned_07_energy_consumption.csv'
df.to_csv(cleaned_filename, index=False)
print(f"Cleaned and enriched dataset saved successfully to: {cleaned_filename} ({df.shape[0]} rows x {df.shape[1]} columns)")

Cleaned and enriched dataset saved successfully to: cleaned_07_energy_consumption.csv (600 rows x 16 columns)


### Five Evidence-Based Energy-Saving Recommendations

1. **Prioritize Energy Audits and Equipment Upgrades in Top-Consuming Buildings (`Block A` and `Lab`):**  
   `Block A` (68,330.10 kWh; 26.51% of total consumption; estimated cost ₹5,80,805.85) and `Lab` (67,600.55 kWh; 26.23% of total consumption; estimated cost ₹5,74,604.68) together account for **52.74%** of total facility energy use and expenditure. Conducting HVAC efficiency tune-ups, LED lighting retrofits, and laboratory equipment power-management audits in these two buildings will deliver the highest energy reduction.

2. **Deploy Occupancy-Driven Smart Controls to Reduce Low-Attendance Base Load (`Block A` & `Block C`):**  
   `Block A` exhibits the highest occupancy-weighted energy intensity at **2.99 kWh/occupant** (record-level mean: 4.73 kWh/occupant), followed by `Block C` at **2.75 kWh/occupant**. Because electrical load remains high even when attendance is low, installing occupancy-based lighting sensors and automated HVAC setback schedules during low-occupancy hours will curb unnecessary baseline consumption.

3. **Shift Flexible Loads from Peak Hours to Off-Peak Windows:**  
   Peak-hour usage accounts for **36.46%** (92,269.50 kWh) of combined metered consumption (`Peak_Hour_Units` + `Off_Peak_Units`), with `Lab` showing the highest peak share at **36.80%**. Rescheduling energy-intensive batch tasks—such as water pumping, battery/EV charging, and non-urgent laboratory testing—to off-peak periods will reduce peak grid stress and commercial Time-of-Day (ToD) surcharges.

4. **Implement Automated Smart-Meter Alerts Above the `500 kWh/day` Threshold:**  
   Out of 600 records, **154 records (25.67%)** exceed the documented `500.0 kWh` threshold and generate **33.30%** (85,814.30 kWh) of total consumption—led by `Lab` (48 high-usage days) and `Block A` (44 high-usage days). Configuring real-time alerts whenever daily consumption crosses 500 kWh enables facility engineers to promptly detect chillers or equipment left running overnight.

5. **Strengthen Sub-Metering Calibration and Automated Data Validation Pipelines:**  
   Ten records (1.67%) in the raw CSV contained missing `Units_kWh` values, and individual records show discrepancies between `Units_kWh` and `Peak_Hour_Units + Off_Peak_Units` (totaling 257,714.05 kWh vs. 253,057.80 kWh) due to un-metered auxiliary circuits or synthetic dataset noise. Upgrading digital sub-meters with automated null-value checks will improve billing accuracy and isolate un-metered losses.

---

### Assumptions & Limitations
- **Synthetic Teaching Dataset:** `07_energy_consumption.csv` is a synthetic teaching dataset (600 records spanning 2025-01-01 to 2026-08-23); patterns illustrate analytical workflows and should not be extrapolated directly to live utility contracts without physical meter verification.
- **Assumed Electricity Tariff:** Cost estimates rely on an explicitly assumed flat commercial rate of **INR 8.50/kWh** (`₹8.50/kWh`); real utility tariffs vary by Time-of-Day (ToD) slabs, fixed demand charges, and tax brackets.
- **Correlation vs. Causation:** Weak Pearson correlations between `Temperature_C` and `Units_kWh` ($r = -0.0697$) and between `Occupancy` and `Units_kWh` ($r = -0.04$) reflect linear association only and do not establish causal relationships.